# Generic search: example

Matched-filter maps over the whole footprint of the Euclid DR1 × DES Y6 stars, to find new streams. For each distance modulus: the stars in the colour-magnitude match filter of an isochrone are counted on HEALPix pixels and smoothed; a smooth 2D polynomial background is fitted and subtracted. A stream shows up in the corrected map.

The notebook runs on a **fake catalog** by default (uniform stars and a fake stream), and on the **Euclid × DES stars** with `USE_FAKE_CATALOG = False`.

Steps:
1. stars and footprint;
2. the search, prepared once;
3. one map: CMD selection, map before correction, fitted background, corrected map;
4. the scan over distance moduli: figures, gif and file of maps (`run_generic_search`).

In [ ]:
from pathlib import Path

import astropy.coordinates as coord
import astropy.units as u
import gala.coordinates as gc
import healpy as hp
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from streameucliddes.analysis.cmd_match_filter import select_stars_in_match_filter
from streameucliddes.analysis.generic_stream_search import (
    generic_search_cmd_stars,
    generic_search_single_dm,
    prepare_generic_search,
)
from streameucliddes.catalogs.euclid_des_catalog import get_eucliddes_stars
from streameucliddes.catalogs.survey_footprint import survey_footprint_healpix_mask
from streameucliddes.plotting import plot_style
from streameucliddes.plotting.plot_generic_search import plot_generic_search_result
from streameucliddes.plotting.plot_sky_density import plot_2D_density_from_hp
from streameucliddes.stream_frames import phi12_to_radec
from streameucliddes.workflows.run_config import GENERIC_SEARCH_CONFIG
from streameucliddes.workflows.run_generic_search import run_generic_search

## 1. Stars and footprint

The Euclid × DES stars. Their magnitudes are already dereddened: there is no dust correction. The match filter reads the `<namespace>_<band>_obs` columns (here Euclid VIS and H, the default match filter).

The fake catalog: stars uniform on the sky and in colour-magnitude space, across ra = 0, and a fake stream along a great circle, at distance modulus 17, its stars inside the match filter of the default isochrone (`select_stars_in_match_filter`).

In [ ]:
USE_FAKE_CATALOG = True  # no Euclid DR1 data yet; False: the Euclid x DES catalog
STREAM_DM = 17.0  # fake stream

if USE_FAKE_CATALOG:
    rng = np.random.default_rng(42)
    N_FAKE_STARS = 3_000_000
    RA_RANGE, DEC_RANGE = (-30.0, 40.0), (-45.0, -5.0)  # deg, across ra = 0
    ra = rng.uniform(*RA_RANGE, N_FAKE_STARS) % 360.0
    dec = np.degrees(np.arcsin(rng.uniform(*np.sin(np.radians(DEC_RANGE)), N_FAKE_STARS)))  # uniform on the sphere
    vis = rng.uniform(18.0, 26.0, N_FAKE_STARS)
    background = pd.DataFrame({"ra": ra, "dec": dec, "euclid_q1_VIS_obs": vis, "euclid_q1_H_obs": vis - rng.uniform(-1.0, 1.5, N_FAKE_STARS)})

    # Fake stream: 40 deg of a great circle, 0.2 deg wide, its stars in the match filter at STREAM_DM
    stream_frame = gc.GreatCircleICRSFrame.from_endpoints(
        coord.SkyCoord(ra=-20.0 * u.deg, dec=-35.0 * u.deg), coord.SkyCoord(ra=25.0 * u.deg, dec=-15.0 * u.deg)
    )
    N_STREAM_CANDIDATES = 60_000
    vis = rng.uniform(18.0, 26.0, N_STREAM_CANDIDATES)
    candidates = pd.DataFrame({"euclid_q1_VIS_obs": vis, "euclid_q1_H_obs": vis - rng.uniform(-1.0, 1.5, N_STREAM_CANDIDATES)})
    stream = candidates[select_stars_in_match_filter(candidates, STREAM_DM)].copy()
    stream["ra"], stream["dec"] = phi12_to_radec(rng.uniform(-20.0, 20.0, len(stream)), rng.normal(0.0, 0.2, len(stream)), stream_frame)
    stream["ra"] %= 360.0

    stars = pd.concat([background, stream], ignore_index=True)
    stars["euclid_q1_VIS_err"] = stars["euclid_q1_H_err"] = 0.02  # small errors: pass the default cuts
    print(f"fake stream: {len(stream):,} stars")

In [ ]:
if not USE_FAKE_CATALOG:
    stars = get_eucliddes_stars()

NSIDE = 256 if USE_FAKE_CATALOG else 512  # maps; 512: pixels of about 0.11 deg
footprint = survey_footprint_healpix_mask(stars["ra"].to_numpy(), stars["dec"].to_numpy(), NSIDE, nest=False)
area = (~footprint).sum() * hp.nside2pixarea(NSIDE, degrees=True)
print(f"{len(stars):,} stars; footprint: {area:,.0f} deg2")
fig, ax = plot_2D_density_from_hp(np.where(footprint, hp.UNSEEN, 1.0), nest=False, cmap="Greys", vmin=0, vmax=2,
                                  title="Footprint", add_colorbar=False)

## 2. The search, prepared once

`prepare_generic_search` checks the columns, applies the cuts (default: errors ≤ 0.2 on the match-filter bands) and finds the HEALPix pixel of each star. The catalog is neither copied nor modified: per star, the search keeps only whether it passes the cuts and its pixel. The maps then read the magnitude columns by chunks.

Defaults: the match filter Euclid VIS vs VIS − H; the isochrone grid of `isochrone_parameters.py` (13 Gyr, Z = 0.0002: one set of maps per isochrone); distance moduli 15 to 22.

In [ ]:
MATCH_FILTER = {  # Euclid VIS vs VIS - H: the default match filter
    "name": "euclid_vis_h",
    "reference_band": ("euclid_q1", "VIS"),
    "color_bands": [("euclid_q1", "H")],
}
SMOOTHING_DEG = 0.3

search = prepare_generic_search(stars, match_filter=MATCH_FILTER, nside=NSIDE, footprint_mask=footprint, nest=False)
print(f"{search.keep.sum():,} stars kept; isochrones {search.isochrones}; "
      f"distance moduli {search.distance_moduli[0]} to {search.distance_moduli[-1]}")

## 3. One map

At one distance modulus: the stars in the match filter (top left: on the CMD of a sample of the stars, `generic_search_cmd_stars`), counted per pixel and smoothed (top right; the smoothed counts are divided by the smoothed footprint: no drop at its edges); the 2D polynomial background, of degree 5, fitted on the longitude around the footprint centre (ra = 0 does not cut it; bottom left); the corrected map, map − background (bottom right).

In [ ]:
result = generic_search_single_dm(distance_modulus=STREAM_DM, search=search, smoothing_deg=SMOOTHING_DEG)
cmd_stars = generic_search_cmd_stars(search)
fig, axes = plot_generic_search_result(result, cmd_stars=cmd_stars)

## 4. Scan over distance moduli: figures, gif and file of maps

`run_generic_search` does step 3 for every distance modulus and isochrone: one figure per distance modulus, their gif, and the corrected maps in one Parquet file (`hpx<nside>` pixels, RING as all the maps of the package, and one column per distance modulus).

Its arguments, with the default values, are in `GENERIC_SEARCH_CONFIG`: copy it and change what you need. Here: a coarser grid of distance moduli, for a quick run. The outputs go to `figures/outputs/` (gitignored).

In [ ]:
config = {
    **GENERIC_SEARCH_CONFIG,
    "match_filter": MATCH_FILTER,
    "nside": NSIDE,
    "smoothing_deg": SMOOTHING_DEG,
    "distance_moduli": np.arange(15.0, 22.0 + 0.25, 0.5),
    "output_dir": "../figures/outputs/generic_search",
    "name": "fake" if USE_FAKE_CATALOG else "eucliddes",
}
summary = run_generic_search(stars, footprint_mask=footprint, nest=False, **config)
for outputs in summary["isochrones"]:
    print(outputs["isochrone"], outputs["gif_path"], outputs["maps_path"], sep="\n  ")

In [ ]:
# The file of maps, read back: one column per distance modulus, the parameters in its metadata
maps_path = summary["isochrones"][0]["maps_path"]
maps = pd.read_parquet(maps_path)
metadata = json.loads(pq.read_schema(maps_path).metadata[b"streameucliddes"])
print({key: metadata[key] for key in ("map_key", "nside", "nest", "isochrone", "smoothing_deg", "polynomial_degree")})
maps.head()